In [ ]:
# Google Colab 환경 설정
!pip install -q rdkit-pypi scikit-learn pandas numpy

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from sklearn.model_selection import cross_validate, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestRegressor

In [ ]:
def remove_collinear_features(X, threshold=0.95):
    df_X = pd.DataFrame(X)
    corr_matrix = df_X.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    to_keep = [c for c in df_X.columns if c not in to_drop]
    return X[:, to_keep]

---
도전 2: δD는 핵심 Descriptor 5개만으로 얼마나 버티나?

Forward Selection으로 탐색한 최적 5개 조합.
예상과 달리 MolWt(분자량)이 아닌 VSA 계열이 선택됨.

분산력(δD)은 분자의 '질량'보다 '부피·표면적'에서 온다는
물리적 직관을 모델이 스스로 찾아낸 결과.

SlogP_VSA5    : LogP 범위별로 분류된 Van der Waals 표면적.
                분자의 소수성 표면 크기를 반영.
                (CV R²: 0.53 → 5개 중 가장 큰 단독 기여)

RingCount     : 고리 구조(벤젠, 사이클로헥산 등) 개수.
                고리가 많을수록 분자가 조밀·부피가 커짐
                → 분산력 ↑

SMR_VSA10     : Molar Refractivity 범위별 표면적.
                분자의 부피와 분극률(polarizability)을 동시에 반영.
                분극률이 클수록 분산력도 큰 경향.

HallKierAlpha : 분자의 형태와 크기를 보정하는 위상적 지수.
                단순 원자 수가 아닌 원자의 크기 차이까지 고려.

VSA_EState8   : 전자 상태(EState) 범위별 표면적.
                분자 내 전자 분포와 표면적을 결합한 지수.
---

In [ ]:
TARGET    = "δD    (MPa0.5)"
KEY_DESCS = [
    'SlogP_VSA5',   # LogP 기반 Van der Waals 표면적
    'RingCount',    # 고리 구조 개수
    'SMR_VSA10',    # Molar Refractivity 기반 표면적
    'HallKierAlpha',# 분자 형태·크기 위상적 지수
    'VSA_EState8',  # EState 기반 표면적
]

In [ ]:
def run(X_final, y, label):
    model = RandomForestRegressor(n_estimators=300, min_samples_leaf=4,
                                  max_samples=0.5, random_state=42, n_jobs=-1)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    s  = cross_validate(model, X_final, y, cv=cv, scoring='r2', return_train_score=True)
    tr, te = s['train_score'].mean(), s['test_score'].mean()
    print(f"  {label:<35} 특성 수: {X_final.shape[1]:>4}   "
          f"Train: {tr:.4f}   CV Test: {te:.4f}   Gap: {tr-te:.4f}")

In [ ]:
print("=" * 70)
print(f"  도전 2: δD — 전체 Descriptor vs 최적 {len(KEY_DESCS)}개")
print(f"  선택된 5개: {KEY_DESCS}")
print("=" * 70)

df   = pd.read_csv('hsp.csv')
mols = [Chem.MolFromSmiles(str(s)) if pd.notnull(s) else None for s in df['Smiles']]

desc_names = [x[0] for x in Descriptors.descList]

In [ ]:
import inspect
    desc_doc = {name: inspect.getdoc(fn) for name, fn in Descriptors.descList}
    name_w, doc_w = 32, 60
    print(f"\n  {'사용 가능한 Descriptor 목록 (KEY_DESCS 참고용)':=<{name_w+doc_w}}")
    print(f"  {'Descriptor':<{name_w}} {'설명':<{doc_w}}")
    print(f"  {'-'*name_w} {'-'*doc_w}")
    for name in desc_names:
        doc = desc_doc.get(name)
        doc_str = doc.split('\n')[0].strip()[:doc_w] if doc else ''
        print(f"  {name:<{name_w}} {doc_str}")
    print(f"  {'':=<{name_w+doc_w}}")
    print(f"  총 {len(desc_names)}개  |  위 이름을 KEY_DESCS 리스트에 넣어보세요\n")
    calc       = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)

    X_raw, valid_indices = [], []
    for i, mol in enumerate(mols):
        if mol:
            try:
                X_raw.append(calc.CalcDescriptors(mol))
                valid_indices.append(i)
            except:
                pass

    y       = df.iloc[valid_indices][TARGET].values
    X_raw   = np.array(X_raw)

In [ ]:
X_imp  = SimpleImputer(strategy='mean').fit_transform(X_raw)
    X_var  = VarianceThreshold(threshold=0.01).fit_transform(X_imp)
    X_sc   = StandardScaler().fit_transform(X_var)
    X_all  = remove_collinear_features(X_sc, threshold=0.95)

In [ ]:
X_imp_full = SimpleImputer(strategy='mean').fit_transform(X_raw)
    key_idx    = [desc_names.index(d) for d in KEY_DESCS]
    X_key      = StandardScaler().fit_transform(X_imp_full[:, key_idx])

    print()
    run(X_all, y, "전체 Descriptor (프루닝 후)")
    run(X_key, y, f"최적 {len(KEY_DESCS)}개")
    print()
    print("  → 분산력(δD)은 부피·표면적(VSA)이 결정적")
    print("  → 5개만으로도 전체 Descriptor와 거의 동일한 성능 (0.8175 vs 0.8150)")
    print()